In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
!pip -q install rasterio segmentation-models-pytorch scikit-image kaggle

## Download DeepGlobe from Kaggle

In [ ]:
import os
import glob
import shutil

candidates = ["kaggle.json", "../kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json")] \
    + glob.glob("/content/**/kaggle.json", recursive=True)
src = next((p for p in candidates if os.path.exists(p)), None)
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
if src:
    shutil.copy(src, os.path.expanduser("~/.kaggle/kaggle.json"))
    print("using kaggle.json from:", src)
else:
    from google.colab import files
    up = files.upload()
    open(os.path.expanduser("~/.kaggle/kaggle.json"), "wb").write(list(up.values())[0])
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

!kaggle datasets download -d balraj98/deepglobe-road-extraction-dataset -p /content/dg
!cd /content/dg && unzip -q -o deepglobe-road-extraction-dataset.zip 'train/*'
print("DeepGlobe train pairs:", len(glob.glob("/content/dg/train/*_sat.jpg")))

## Upload the four Sentinel-2 files

In [ ]:
from google.colab import files
os.makedirs("/content/wk2", exist_ok=True)
for name, data in files.upload().items():
    open(f"/content/wk2/{name}", "wb").write(data)
print("have:", sorted(os.listdir("/content/wk2")))

## Build DeepGlobe patches (downsample 16x to ~8 m)

In [ ]:
import numpy as np
from PIL import Image

DG_OUT = "/content/patches/deepglobe"
FACTOR = 16
PATCH = 1024 // FACTOR
for sub in ("train", "val"):
    os.makedirs(f"{DG_OUT}/{sub}", exist_ok=True)


def downsample(img, mask):
    nh, nw = img.shape[0] // FACTOR, img.shape[1] // FACTOR
    img = np.asarray(Image.fromarray(img).resize((nw, nh), Image.BILINEAR), dtype="float32") / 255.0
    mask = (np.asarray(Image.fromarray(mask).resize((nw, nh), Image.NEAREST)) > 127).astype("uint8")
    return img, mask


sats = sorted(glob.glob("/content/dg/train/*_sat.jpg"))
rng = np.random.default_rng(0)
counts = {"train": 0, "val": 0}
for n, sat in enumerate(sats):
    mask_path = sat.replace("_sat.jpg", "_mask.png")
    if not os.path.exists(mask_path):
        continue
    img = np.asarray(Image.open(sat).convert("RGB"))
    mask = np.asarray(Image.open(mask_path).convert("L"))
    img_s, mask_s = downsample(img, mask)
    if mask_s.sum() == 0 and rng.random() > 0.15:
        continue
    sub = "val" if n % 10 == 0 else "train"
    chw = np.transpose(img_s, (2, 0, 1))
    np.savez_compressed(f"{DG_OUT}/{sub}/d{n:05d}.npz", img=chw.astype("float32"), mask=mask_s)
    counts[sub] += 1
print("deepglobe patches:", counts)

## Build the local Sentinel-2 patches

In [ ]:
import rasterio

LC_OUT = "/content/patches/local"
for sub in ("train", "val"):
    os.makedirs(f"{LC_OUT}/{sub}", exist_ok=True)
PATCH, STRIDE, VAL_FRAC = 256, 128, 0.28


def stretch(arr):
    out = np.empty_like(arr, dtype="float32")
    for b in range(arr.shape[0]):
        finite = arr[b][np.isfinite(arr[b])]
        lo, hi = np.percentile(finite, 2), np.percentile(finite, 98)
        out[b] = np.clip((arr[b] - lo) / (hi - lo + 1e-6), 0, 1)
    return np.nan_to_num(out)


with rasterio.open("/content/wk2/stack_2024.tif") as s:
    img = stretch(s.read([3, 2, 1, 4, 5]).astype("float32"))
    H, W = s.height, s.width
with rasterio.open("/content/wk2/osm_roads_2024_mask.tif") as r:
    mask = (r.read(1) == 1).astype("uint8")

split = int(W * (1 - VAL_FRAC))
counts = {"train": 0, "val": 0}
for y in range(0, H - PATCH + 1, STRIDE):
    for x in range(0, W - PATCH + 1, STRIDE):
        if x >= split:
            sub = "val"
        elif x + PATCH <= split:
            sub = "train"
        else:
            continue
        patch = img[:, y:y+PATCH, x:x+PATCH]
        patch_mask = mask[y:y+PATCH, x:x+PATCH]
        if patch[:3].max() < 1e-4:
            continue
        np.savez_compressed(f"{LC_OUT}/{sub}/p_{y}_{x}.npz", img=patch.astype("float32"), mask=patch_mask)
        counts[sub] += 1
print("local patches:", counts)

## U-Net, dataset and training loop

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=64):
        super().__init__()
        self.e1 = DoubleConv(in_ch, base)
        self.e2 = DoubleConv(base, base * 2)
        self.e3 = DoubleConv(base * 2, base * 4)
        self.e4 = DoubleConv(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.bott = DoubleConv(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, 2, 2)
        self.d4 = DoubleConv(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.d3 = DoubleConv(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.d2 = DoubleConv(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.d1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.bott(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.head(d1)


class PatchDS(Dataset):
    def __init__(self, folder, channels=3, augment=False):
        self.files = sorted(glob.glob(f"{folder}/*.npz"))
        self.channels = channels
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def _augment(self, img, mask):
        if np.random.rand() < 0.5:
            img, mask = img[:, :, ::-1], mask[:, ::-1]
        if np.random.rand() < 0.5:
            img, mask = img[:, ::-1, :], mask[::-1, :]
        k = np.random.randint(4)
        if k:
            img, mask = np.rot90(img, k, (1, 2)), np.rot90(mask, k)
        if np.random.rand() < 0.5:
            img = np.clip(img * np.random.uniform(0.85, 1.15), 0, 1)
        return np.ascontiguousarray(img), np.ascontiguousarray(mask)

    def __getitem__(self, i):
        data = np.load(self.files[i])
        img, mask = data["img"][:self.channels], data["mask"]
        if self.augment:
            img, mask = self._augment(img, mask)
        return torch.from_numpy(img.astype("float32")), torch.from_numpy(mask.astype("float32"))[None]


def dice_bce(logits, target, eps=1.0):
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    p = torch.sigmoid(logits)
    inter = (p * target).sum((1, 2, 3))
    dice = 1 - (2 * inter + eps) / (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + eps)
    return bce + dice.mean()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tp = fp = fn = 0
    for x, y in loader:
        p = (torch.sigmoid(model(x.to(DEVICE))) > 0.5).float().cpu()
        tp += float((p * y).sum())
        fp += float((p * (1 - y)).sum())
        fn += float(((1 - p) * y).sum())
    return tp / (tp + fp + fn + 1e-6), 2 * tp / (2 * tp + fp + fn + 1e-6)


def build(name):
    if name == "smp":
        import segmentation_models_pytorch as smp
        return smp.Unet("resnet34", encoder_weights="imagenet", in_channels=3, classes=1)
    return UNet(in_ch=3)


def train(model, data, epochs, lr, out, batch=16):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = DataLoader(PatchDS(f"{data}/train", 3, augment=True), batch_size=batch, shuffle=True)
    val_loader = DataLoader(PatchDS(f"{data}/val", 3), batch_size=batch)
    best = 0
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = dice_bce(model(x), y)
            loss.backward()
            opt.step()
            total += loss.item() * x.size(0)
        iou, f1 = evaluate(model, val_loader)
        flag = ""
        if iou > best:
            best = iou
            torch.save(model.state_dict(), out)
            flag = " <- best"
        print(f"  epoch {epoch:2d} loss {total/len(train_loader.dataset):.4f} val IoU {iou:.3f} F1 {f1:.3f}{flag}")
    print(f"  best IoU {best:.3f} -> {out}")
    return best


print("device:", DEVICE)

## Train both models (scratch: DeepGlobe pretrain then fine-tune; ResNet34: fine-tune from ImageNet)

In [ ]:
os.makedirs("/content/out", exist_ok=True)

print("scratch: pretrain on DeepGlobe")
model = build("scratch")
train(model, "/content/patches/deepglobe", epochs=12, lr=1e-3, out="/content/out/scratch_pre.pt")

print("scratch: fine-tune on local")
model = build("scratch")
model.load_state_dict(torch.load("/content/out/scratch_pre.pt"))
train(model, "/content/patches/local", epochs=40, lr=3e-4, out="/content/out/scratch_ft.pt", batch=8)

print("resnet34: fine-tune from ImageNet")
model = build("smp")
train(model, "/content/patches/local", epochs=40, lr=3e-4, out="/content/out/smp_ft.pt", batch=8)

## Comparison table (held-out local val)

In [ ]:
def score(ckpt, name):
    model = build("smp" if "smp" in name else "scratch").to(DEVICE)
    model.load_state_dict(torch.load(ckpt))
    model.eval()
    val_loader = DataLoader(PatchDS("/content/patches/local/val", 3), batch_size=8)
    tp = fp = fn = 0
    with torch.no_grad():
        for x, y in val_loader:
            p = (torch.sigmoid(model(x.to(DEVICE))) > 0.5).float().cpu()
            tp += float((p * y).sum())
            fp += float((p * (1 - y)).sum())
            fn += float(((1 - p) * y).sum())
    iou = tp / (tp + fp + fn + 1e-6)
    f1 = 2 * tp / (2 * tp + fp + fn + 1e-6)
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    print(f"{name:18s} IoU {iou:.3f}  F1 {f1:.3f}  Prec {precision:.3f}  Recall {recall:.3f}")


print(f"{'model':18s} metrics")
print("-" * 55)
score("/content/out/scratch_ft.pt", "scratch_finetuned")
score("/content/out/smp_ft.pt", "smp_finetuned")

## Inference over the full 2017 and 2024 images

In [ ]:
P, STR = 256, 192


def infer(stack_path, ckpt, name, out_path):
    model = build("smp" if "smp" in name else "scratch").to(DEVICE)
    model.load_state_dict(torch.load(ckpt))
    model.eval()
    with rasterio.open(stack_path) as s:
        img = stretch(s.read([3, 2, 1, 4, 5]).astype("float32"))[:3]
        profile = s.profile
        H, W = s.height, s.width
    prob = np.zeros((H, W), "float32")
    cover = np.zeros((H, W), "float32")
    ys = sorted(set(list(range(0, max(H - P, 0) + 1, STR)) + [H - P]))
    xs = sorted(set(list(range(0, max(W - P, 0) + 1, STR)) + [W - P]))
    with torch.no_grad():
        for y in ys:
            for x in xs:
                t = torch.from_numpy(img[:, y:y+P, x:x+P][None]).to(DEVICE)
                prob[y:y+P, x:x+P] += torch.sigmoid(model(t)).cpu().numpy()[0, 0]
                cover[y:y+P, x:x+P] += 1
    prob /= np.maximum(cover, 1)
    profile.update(count=1, dtype="float32", nodata=None)
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(prob.astype("float32"), 1)
    print(f"{out_path}: >0.5 covers {100*(prob>0.5).mean():.1f}%")


infer("/content/wk2/stack_2017.tif", "/content/out/scratch_ft.pt", "scratch", "/content/out/pred_2017.tif")
infer("/content/wk2/stack_2024.tif", "/content/out/scratch_ft.pt", "scratch", "/content/out/pred_2024.tif")

## Quantify road growth and figure

In [ ]:
from skimage.morphology import skeletonize
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


def read(path):
    with rasterio.open(path) as s:
        return s.read(1)


def km_length(mask):
    return int(skeletonize(mask).sum()) * 10 / 1000


def km2_area(mask):
    return int(mask.sum()) * 100 / 1e6


p17 = read("/content/out/pred_2017.tif") > 0.5
p24 = read("/content/out/pred_2024.tif") > 0.5
new = p24 & ~p17
print(f"predicted 2017: {km2_area(p17):.2f} km^2  {km_length(p17):.0f} km")
print(f"predicted 2024: {km2_area(p24):.2f} km^2  {km_length(p24):.0f} km")
print(f"new since 2017:  {km2_area(new):.2f} km^2  {km_length(new):.0f} km")
o24 = read("/content/wk2/osm_roads_2024_mask.tif") == 1
print(f"model-vs-OSM 2024 IoU: {int((p24 & o24).sum())/int((p24 | o24).sum()):.3f}")

with rasterio.open("/content/wk2/stack_2024.tif") as s:
    rgb = np.transpose(stretch(s.read([3, 2, 1]).astype("float32")), (1, 2, 0))
overlay = rgb.copy()
overlay[p17] = [0, 0.45, 1]
overlay[new] = [1, 0, 0]
plt.figure(figsize=(10, 9))
plt.imshow(overlay)
plt.xticks([])
plt.yticks([])
plt.title("U-Net road growth: 2017 (blue) vs new by 2024 (red)")
plt.legend(handles=[Patch(color=[0, 0.45, 1], label="2017"), Patch(color=[1, 0, 0], label="new by 2024")],
           loc="lower right")
plt.savefig("/content/out/pred_growth.png", dpi=140, bbox_inches="tight")
plt.show()

## Download the results

In [ ]:
from google.colab import files
for f in ["scratch_ft.pt", "smp_ft.pt", "pred_2017.tif", "pred_2024.tif", "pred_growth.png"]:
    files.download(f"/content/out/{f}")